In [14]:
from transformers import CLIPProcessor, CLIPModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,precision_score, recall_score, classification_report
from PIL import Image, UnidentifiedImageError
import torch
import numpy as np
import pandas as pd


In [2]:
##------------Loading the clip model and processor----------------##
#define the device to use for the model
# If a GPU is available, use it; otherwise, use the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the CLIP model and processor needed for feature extraction
# The model is loaded to the specified device (GPU or CPU)
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

# Load the processor for preparing inputs to the model
# The processor handles the preprocessing of images and text for the CLIP model
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
##------------Loading and processing the dataset----------------##
df = pd.read_csv("Complete_Img.csv")  

# Initialize lists to hold image embeddings and labels
image_text_embeddings = []
labels = []

In [4]:
##-----------------------Feature Extraction----------------##
#we need feature extraction to convert images and text into embeddings

# Iterate through each row in the DataFrame
# Each row contains an image path, text, and label
for idx, row in df.iterrows():
    image_path = row["image"] # Ensure the image path is a string
    text = str(row["text"]) if pd.notnull(row["text"]) else "" # Ensure text is a string, default to empty if NaN
    label = row["label"] # Ensure label is a string or numeric value

    # Check if the image path is a valid string and the file exists
    #need to handle cases where the image path might be invalid or the file does not exist
    try:
        image = Image.open(image_path).convert("RGB")# Convert image to RGB format

        # Prepare inputs for the model using the processor, which handles both text and image inputs
        # The processor will ensure that the text is a string and the image is in the correct
        inputs = processor(
            text=[text],  # text should be a list, even if it's a single string
            images=image, # image should be a PIL Image object
            return_tensors="pt", # Return PyTorch tensors
            padding=True, # Pad the inputs to the maximum length
            truncation=True # Truncate the text if it exceeds the maximum length
        ).to(device)

        #with torch.no_grad() is used to disable gradient calculation, which is not needed during inference
        # This saves memory and speeds up computations, with means we are not training the model, just extracting features
        with torch.no_grad():
            outputs = model(**inputs) # Extract features from the model
            image_emb = outputs.image_embeds[0] # Get the image embeddings
            text_emb = outputs.text_embeds[0] # Get the text embeddings
            combined_emb = torch.cat([image_emb, text_emb]).cpu().numpy() # Combine image and text embeddings into a single vector
        # Append the combined embeddings and label to the respective lists
        image_text_embeddings.append(combined_emb)
        labels.append(label)
    # Handle exceptions for file not found, image loading errors, or other issues
    except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError) as e:
        print(f"Skipping row {idx}: {e}")
        continue

#convert lists to numpy arrays, needed for sklearn to train the model because sklearn expects numpy arrays or pandas dataframes
X = np.array(image_text_embeddings)
y = np.array(labels)

C:\Users\nutme\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\PIL\Image.py:3570: UserWarning: image file could not be identified because AVIF support not installed
  warnings.warn(message)


Skipping row 35: cannot identify image file 'images/IHS_0542.jpg'
Skipping row 48: cannot identify image file 'images/IHS_0123.jpg'
Skipping row 194: cannot identify image file 'images/IHS_0607.jpg'
Skipping row 314: cannot identify image file 'images/IHS_0525.jpg'
Skipping row 349: cannot identify image file 'images/IHS_0083.jpg'
Skipping row 467: cannot identify image file 'images/IHS_0610.jpg'
Skipping row 494: cannot identify image file 'images/IHS_0446.jpg'
Skipping row 623: cannot identify image file 'images/IHS_0381.jpg'
Skipping row 771: cannot identify image file 'images/IHS_0367.jpg'
Skipping row 859: cannot identify image file 'images/IHS_0524.jpg'
Skipping row 1300: cannot identify image file 'images/IHS_0288.jpg'
Skipping row 1403: cannot identify image file 'images/IHS_0468.jpg'
Skipping row 1453: cannot identify image file 'images/IHS_0251.jpg'
Skipping row 2126: cannot identify image file 'images/IHS_0118.jpg'
Skipping row 2318: cannot identify image file 'images/IHS_05

In [7]:
##-----------------------Train-Validation-Test Split----------------##
# Split the dataset into training, validation, and test sets
#print out each split size

# split into train (70%) and temp (30%)
#stratify ensures that the class distribution is maintained in each split, so that the training, validation, and test sets have the same proportion of classes as the original dataset
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=50, stratify=y)

# Now split temp by 50% into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=50, stratify=y_temp)

print("Train set size:", len(X_train), "Validation set size:", len(X_val),
      "Test set size:", len(y_val))

Train set size: 2694 Validation set size: 577 Test set size: 577


In [ ]:
##------------------------Train and Evaulation (LR)--------------------------------##
#logistic regression model for classification, we use this because it is simple and effective for binary classification tasks
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)

y_pred = clf.predict(X_test)

f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(classification_report(y_test, y_pred, digits=4))

In [10]:
##------------------------Train and Evaulation (RF)--------------------------------##
#Random Forest model for classification
clf = RandomForestClassifier(n_estimators=250).fit(X_train, y_train)

y_pred = clf.predict(X_test)

f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(classification_report(y_test, y_pred, digits=4))

Precision: 0.8997
Recall:    0.8997
F1 Score:  0.8996
                 precision    recall  f1-score   support

    Hope_speech     0.9046    0.8920    0.8982       287
Non_hope_speech     0.8949    0.9072    0.9010       291

       accuracy                         0.8997       578
      macro avg     0.8998    0.8996    0.8996       578
   weighted avg     0.8997    0.8997    0.8996       578



In [ ]:
##------------------------Train and Evaulation (XGB)--------------------------------##
#XGBoost model for classification
#need to translate hope and non hope into 1,0 

#needed to convert labels
labelEncoder = LabelEncoder()

#new variables that will store converted labels
y_trainE =labelEncoder.fit_transform(y_train)
y_testE = labelEncoder.transform(y_test)

clf = XGBClassifier(n_estimators=250).fit(X_train, y_trainE)

y_predE = clf.predict(X_test)

f1 = f1_score(y_testE, y_predE, average='weighted')
precision = precision_score(y_testE, y_predE, average='weighted')
recall = recall_score(y_testE, y_predE, average='weighted')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(classification_report(y_testE, y_predE, digits=4))

Precision: 0.9031
Recall:    0.9031
F1 Score:  0.9031
              precision    recall  f1-score   support

           0     0.9024    0.9024    0.9024       287
           1     0.9038    0.9038    0.9038       291

    accuracy                         0.9031       578
   macro avg     0.9031    0.9031    0.9031       578
weighted avg     0.9031    0.9031    0.9031       578

